In [ ]:

!pip install matplotlib numpy --quiet

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Dark background for all plots
plt.style.use('dark_background')

In [ ]:

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_facecolor('#000005')
ax.set_xlim(-10, 10)
ax.set_ylim(-10, 10)
ax.set_aspect('equal')
ax.axis('off')

# The Schwarzschild radius — this is the real formula
# In our simulation units: r_s = 2 * M (we set G=c=1)
M = 1.0          # black hole mass
r_s = 2 * M      # event horizon radius
r_photon = 1.5 * r_s   # photon sphere — where light orbits

# Glow rings (drawn first so they appear behind)
for i in range(8, 0, -1):
    glow = plt.Circle((0, 0), r_photon * (1 + i * 0.2),
                       color='darkorange', alpha=0.015 * i, zorder=1)
    ax.add_patch(glow)

# Photon sphere ring
photon_ring = plt.Circle((0, 0), r_photon,
                          fill=False, edgecolor='orange',
                          linewidth=1.5, alpha=0.4, zorder=2)
ax.add_patch(photon_ring)

# Event horizon — the black hole itself
black_hole = plt.Circle((0, 0), r_s,
                         color='black', zorder=3)
ax.add_patch(black_hole)

# Labels
ax.text(r_photon + 0.2, 0.2, 'photon sphere\nr = 1.5 × rₛ',
        color='orange', fontsize=9, alpha=0.7)
ax.text(0.1, -r_s * 0.4, 'event horizon\nr = 2GM/c²',
        color='gray', fontsize=9, ha='center')

ax.set_title('Black Hole — Schwarzschild Geometry',
             color='white', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

print(f"Schwarzschild radius: r_s = {r_s}")
print(f"Photon sphere:        r_ph = {r_photon}")

In [ ]:

from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(7, 7))
ax.set_facecolor('#000005')
ax.set_xlim(-10, 10)
ax.set_ylim(-10, 10)
ax.set_aspect('equal')
ax.axis('off')


M = 1.0
r_s = 2 * M
for i in range(6, 0, -1):
    glow = plt.Circle((0,0), r_s*1.5*(1+i*0.2),
                       color='darkorange', alpha=0.018*i)
    ax.add_patch(glow)
ax.add_patch(plt.Circle((0,0), r_s, color='black', zorder=5))

# One particle orbiting at radius r=5
orbit_r = 5.0
dot, = ax.plot([], [], 'o', color='cyan', ms=6, zorder=6)
trail, = ax.plot([], [], '-', color='cyan', lw=1, alpha=0.4, zorder=5)

# Store trail history
trail_x, trail_y = [], []
trail_length = 40   # how many past positions to show

def init():
    dot.set_data([], [])
    trail.set_data([], [])
    return dot, trail

def update(frame):
    # Keplerian angular speed: omega = sqrt(M / r^3)
    # This is Newton's law — faster orbit = smaller radius
    omega = np.sqrt(M / orbit_r**3)
    angle = frame * omega * 0.3   # 0.3 controls animation speed

    x = orbit_r * np.cos(angle)
    y = orbit_r * np.sin(angle)

    trail_x.append(x)
    trail_y.append(y)

    # Keep only the last N positions
    if len(trail_x) > trail_length:
        trail_x.pop(0)
        trail_y.pop(0)

    dot.set_data([x], [y])
    trail.set_data(trail_x, trail_y)
    return dot, trail

anim = FuncAnimation(fig, update, frames=300,
                     init_func=init, interval=30, blit=True)
plt.tight_layout()

HTML(anim.to_jshtml())

In [ ]:

from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')

M   = 1.0
r_s = 2 * M                          # Schwarzschild radius
r_isco = 3 * r_s                     # Innermost Stable Circular Orbit
#   Inside r_isco, no stable orbit exists — particles spiral in

N = 150   # number of particles

# Give each particle a random starting radius (outside ISCO)
# and a random starting angle
np.random.seed(42)
radii  = np.random.uniform(r_isco, r_isco * 5, N)
angles = np.random.uniform(0, 2 * np.pi, N)

# Each particle slowly drifts inward (accretion)
infall_rate = 0.002   # how fast they spiral in

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_facecolor('#000005')
ax.set_xlim(-14, 14)
ax.set_ylim(-14, 14)
ax.set_aspect('equal')
ax.axis('off')
fig.tight_layout()

# ── Draw static black hole ───────────────────────────────
for i in range(7, 0, -1):
    ax.add_patch(plt.Circle((0,0), r_s*1.5*(1+i*0.25),
                             color='darkorange', alpha=0.014*i, zorder=1))
ax.add_patch(plt.Circle((0,0), r_s, color='black', zorder=10))

# ISCO dashed ring — inside here, doom
isco_ring = plt.Circle((0,0), r_isco, fill=False,
                        linestyle='--', edgecolor='red',
                        linewidth=0.8, alpha=0.4, zorder=2)
ax.add_patch(isco_ring)
ax.text(r_isco * 0.72, r_isco * 0.72, 'ISCO',
        color='red', fontsize=8, alpha=0.6)

# Create particle dots
dots, = ax.plot([], [], 'o', color='orange',
                ms=2.5, zorder=5, alpha=0.9)

frame_count = [0]

def init():
    dots.set_data([], [])
    return dots,

def update(frame):
    frame_count[0] += 1

    # Update each particle's angle (Keplerian orbit)
    # and slowly reduce its radius (inspiral)
    for i in range(N):
        omega = np.sqrt(M / max(radii[i], 0.1)**3)
        angles[i] += omega * 0.25

        # Slow inward drift
        radii[i] -= infall_rate * (r_isco / radii[i])

        # If particle crosses event horizon, respawn at outer edge
        if radii[i] < r_s:
            radii[i] = np.random.uniform(r_isco * 3, r_isco * 5)
            angles[i] = np.random.uniform(0, 2 * np.pi)

    # Flatten disk (squish y axis) — disk is tilted in our view
    x = radii * np.cos(angles)
    y = radii * np.sin(angles) * 0.35   # 0.35 = tilt factor

    dots.set_data(x, y)
    return dots,

anim = FuncAnimation(fig, update, frames=400,
                     init_func=init, interval=30, blit=True)

HTML(anim.to_jshtml())

In [ ]:

from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plt.style.use('dark_background')

M      = 1.0
r_s    = 2 * M
r_isco = 3 * r_s
N      = 120
TRAIL  = 18     # trail length per particle

np.random.seed(7)
radii  = np.random.uniform(r_isco * 1.1, r_isco * 5, N)
angles = np.random.uniform(0, 2 * np.pi, N)

# Store trail history for every particle
history_x = [[] for _ in range(N)]
history_y = [[] for _ in range(N)]

fig, ax = plt.subplots(figsize=(9, 9))
ax.set_facecolor('#000005')
ax.set_xlim(-16, 16)
ax.set_ylim(-16, 16)
ax.set_aspect('equal')
ax.axis('off')

# Black hole glow
for i in range(8, 0, -1):
    ax.add_patch(plt.Circle((0,0), r_s*1.5*(1+i*0.22),
                             color='darkorange', alpha=0.013*i, zorder=1))
ax.add_patch(plt.Circle((0,0), r_s, color='black', zorder=10))

# All trail lines stored here — one per particle
lines = []
for _ in range(N):
    ln, = ax.plot([], [], '-', lw=1.2, alpha=0.0, zorder=4)
    lines.append(ln)

# All particle dots
dots_list = []
for _ in range(N):
    d, = ax.plot([], [], 'o', ms=2.8, zorder=6)
    dots_list.append(d)

def heat_color(r):
    """
    Returns a color based on how close the particle is to the black hole.
    Close = white/yellow (very hot), far = deep red/orange (cooler).
    This mirrors real accretion disk physics — friction heats the gas
    as it falls inward.
    """
    # Normalized distance: 0 = at ISCO, 1 = far away
    t = np.clip((r - r_isco) / (r_isco * 4), 0, 1)
    # Color ramp: white → yellow → orange → deep red
    if t < 0.25:
        return (1.0, 1.0, 0.6 + 0.4*t/0.25)       # white → yellow
    elif t < 0.6:
        tt = (t - 0.25) / 0.35
        return (1.0, 1.0 - 0.5*tt, 0.0)            # yellow → orange
    else:
        tt = (t - 0.6) / 0.4
        return (1.0 - 0.4*tt, 0.2 - 0.15*tt, 0.0) # orange → dark red

def update(frame):
    for i in range(N):
        omega = np.sqrt(M / max(radii[i], 0.1)**3)
        angles[i] += omega * 0.22

        # Gradual infall — stronger near ISCO
        radii[i] -= 0.0018 * (r_isco / max(radii[i], r_s))

        if radii[i] < r_s * 0.95:
            radii[i] = np.random.uniform(r_isco * 2.5, r_isco * 5)
            angles[i] = np.random.uniform(0, 2 * np.pi)
            history_x[i].clear()
            history_y[i].clear()

        x = radii[i] * np.cos(angles[i])
        y = radii[i] * np.sin(angles[i]) * 0.35

        history_x[i].append(x)
        history_y[i].append(y)
        if len(history_x[i]) > TRAIL:
            history_x[i].pop(0)
            history_y[i].pop(0)

        c = heat_color(radii[i])

        # Draw trail (faded)
        lines[i].set_data(history_x[i], history_y[i])
        lines[i].set_color(c)
        lines[i].set_alpha(0.35)

        # Draw dot (bright)
        dots_list[i].set_data([x], [y])
        dots_list[i].set_color(c)
        dots_list[i].set_alpha(0.95)

    return lines + dots_list

anim = FuncAnimation(fig, update, frames=500,
                     init_func=lambda: lines + dots_list,
                     interval=28, blit=True)

HTML(anim.to_jshtml())

In [ ]:
import numpy as np

# ── Constants (natural units: G = c = 1) ──────────────────
# This is standard in theoretical physics. It means:
# - distance is measured in "gravitational radii" (GM/c²)
# - time is measured in units where light travels 1 unit per step
# Setting G=c=1 removes clutter and keeps the math clean.

G = 1.0
c = 1.0

def schwarzschild_radius(M):
    """
    r_s = 2GM/c²
    The event horizon. Anything crossing this radius
    cannot send information back out — not even light.
    For the Sun: r_s ≈ 3 km. For Earth: r_s ≈ 9 mm.
    """
    return 2 * G * M / c**2

def isco_radius(M, a=0.0):
    """
    ISCO = Innermost Stable Circular Orbit.

    For a NON-spinning black hole (a=0): r_isco = 6M = 3 * r_s
    For a MAX-spinning black hole (a=1): r_isco = r_s (prograde orbit)

    'a' is the spin parameter, ranging from 0 (no spin) to 1 (max spin).
    It comes from the Kerr metric — the GR solution for rotating black holes.

    The formula below is an approximation of the exact Kerr ISCO.
    The real formula involves solving a 6th-degree polynomial!
    """
    r_s = schwarzschild_radius(M)
    # Spin shifts ISCO inward: more spin = smaller ISCO = more accretion
    return r_s * (3 + 2*a*(1 - 0.5*a))   # approximation of Kerr ISCO

def photon_sphere_radius(M):
    """
    r_ph = 3GM/c² = 1.5 * r_s
    The radius where photons travel in (unstable) circular orbits.
    This is what creates the bright ring around black holes in real images.
    In the famous M87* image by the Event Horizon Telescope,
    the bright ring IS the photon sphere.
    """
    return 3 * G * M / c**2

def orbital_speed(M, r):
    """
    v = sqrt(GM/r)  — the Newtonian circular orbit speed.

    At the ISCO this approaches a significant fraction of c.
    In full GR the formula is more complex, but this is a
    good approximation outside the photon sphere.
    """
    return np.sqrt(G * M / np.maximum(r, 1e-6))

def keplerian_omega(M, r):
    """
    Angular velocity omega = sqrt(GM / r³)
    This is Kepler's third law: T² ∝ r³
    Closer to the black hole = smaller r = larger omega = faster orbit.
    At r = r_isco the orbital period is just minutes for stellar black holes.
    """
    return np.sqrt(G * M / np.maximum(r, 1e-6)**3)

def gravitational_time_dilation(r, M):
    """
    How much slower time runs near a massive object.
    From the Schwarzschild metric: dt_proper = dt_far * sqrt(1 - r_s/r)

    At r = r_s: time dilation = 0  → time stops completely (for distant observer)
    At r = 2*r_s: time runs at 71% of far-away time
    At r = 10*r_s: time runs at 95% of far-away time

    This is real and measurable — GPS satellites need GR corrections
    or your phone maps would drift by kilometers per day.
    """
    r_s = schwarzschild_radius(M)
    return np.sqrt(np.maximum(0, 1 - r_s / np.maximum(r, r_s * 1.001)))

def disk_temperature(r, M):
    """
    Temperature profile of an accretion disk.
    Based on the Novikov-Thorne model (1973) — the standard model
    for thin accretion disks around black holes.

    T(r) ∝ M^(1/4) * r^(-3/4) * f(r)^(1/4)
    where f(r) is a correction factor near the ISCO.

    Returns a normalized 0→1 value for color mapping.
    """
    r_isco = isco_radius(M)
    if r <= r_isco:
        return 1.0   # maximum temperature at ISCO
    # Temperature drops as r^(-3/4) outside ISCO
    return (r_isco / r) ** 0.75

M = 10.0   # 10 solar masses — a typical stellar black hole

r_s   = schwarzschild_radius(M)
r_isco = isco_radius(M, a=0.0)
r_isco_spin = isco_radius(M, a=0.99)
r_ph  = photon_sphere_radius(M)
v_isco = orbital_speed(M, r_isco)
td    = gravitational_time_dilation(r_isco, M)

print("=" * 50)
print(f"  Black hole mass:          M  = {M} M☉")
print(f"  Schwarzschild radius:     r_s    = {r_s:.2f}")
print(f"  Photon sphere:            r_ph   = {r_ph:.2f}")
print(f"  ISCO (no spin, a=0):      r_isco = {r_isco:.2f}")
print(f"  ISCO (max spin, a=0.99):  r_isco = {r_isco_spin:.2f}")
print(f"  Orbital speed at ISCO:    v      = {v_isco:.3f} c")
print(f"  Time dilation at ISCO:    dt     = {td:.3f}×")
print("=" * 50)
print()
print("Interpretation:")
print(f"  At the ISCO, time runs {td:.1%} as fast as far away.")
print(f"  Orbital speed at ISCO is {v_isco:.1%} of light speed.")
print(f"  Spinning black hole has ISCO {r_isco/r_isco_spin:.1f}× closer in.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact, interactive

plt.style.use('dark_background')



def draw_black_hole_snapshot(
    mass=widgets.FloatSlider(min=0.5, max=5.0, step=0.1, value=1.5,
                             description='Mass M'),
    spin=widgets.FloatSlider(min=0.0, max=0.99, step=0.01, value=0.3,
                             description='Spin a'),
    n_particles=widgets.IntSlider(min=10, max=200, step=10, value=80,
                                  description='Particles')
):
    """
    This function runs every time you move a slider.
    @interact calls it automatically with the new values.
    """
    # ── Compute GR quantities from your Step 5 functions ──
    r_s      = schwarzschild_radius(mass)
    r_isco   = isco_radius(mass, spin)
    r_ph     = photon_sphere_radius(mass)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_facecolor('#000008')
    ax.set_xlim(-12, 12)
    ax.set_ylim(-12, 12)
    ax.set_aspect('equal')
    ax.axis('off')

    # ── Accretion disk glow (ellipses = tilted disk) ───────
    # We draw a series of concentric ellipses getting fainter
    # outward. The y-axis is squished by 0.35 to simulate
    # viewing the disk at an angle from above.
    for i in range(12, 0, -1):
        rx = r_isco + i * r_s * 0.6
        ry = rx * 0.35
        alpha = 0.06 * (1 - i/12)
        disk = patches.Ellipse((0, 0), rx*2, ry*2,
                                color='darkorange', alpha=alpha, zorder=2)
        ax.add_patch(disk)

    # ── Event horizon glow ─────────────────────────────────
    for i in range(6, 0, -1):
        glow = plt.Circle((0, 0), r_ph * (1 + i * 0.25),
                           color='#ff6600', alpha=0.018 * i, zorder=3)
        ax.add_patch(glow)

    # ── Photon sphere ──────────────────────────────────────
    ax.add_patch(plt.Circle((0, 0), r_ph, fill=False,
                             edgecolor='orange', lw=1.5, alpha=0.5, zorder=4))

    # ── ISCO ring ──────────────────────────────────────────
    # Drawing it as an ellipse because the disk is tilted
    isco_el = patches.Ellipse((0, 0), r_isco*2, r_isco*2*0.35,
                               fill=False, linestyle='--',
                               edgecolor='red', lw=0.8, alpha=0.5, zorder=4)
    ax.add_patch(isco_el)

    # ── Scatter particles in the disk ─────────────────────
    # Random radii between ISCO and 5×ISCO
    np.random.seed(42)
    r_pts   = np.random.uniform(r_isco, r_isco * 5, n_particles)
    a_pts   = np.random.uniform(0, 2*np.pi, n_particles)

    # Color each particle by its temperature (radius → heat)
    temps   = np.array([disk_temperature(r, mass) for r in r_pts])

    x_pts   = r_pts * np.cos(a_pts)
    y_pts   = r_pts * np.sin(a_pts) * 0.35   # disk tilt

    # 'afmhot' is a colormap that goes black→red→orange→yellow→white
    cmap    = plt.cm.afmhot
    colors  = cmap(temps)

    ax.scatter(x_pts, y_pts, c=colors, s=4, alpha=0.8, zorder=5)

    # Black hole itself
    ax.add_patch(plt.Circle((0, 0), r_s, color='black', zorder=10))


    # This is the "head-up display" — live numbers from GR formulas
    td  = gravitational_time_dilation(r_isco, mass)
    v   = orbital_speed(mass, r_isco)
    info = (
        f"M = {mass:.1f}   spin a = {spin:.2f}\n"
        f"r_s = {r_s:.2f}   r_isco = {r_isco:.2f}   r_ph = {r_ph:.2f}\n"
        f"v at ISCO = {v:.3f}c   time dilation = {td:.3f}×"
    )
    ax.text(0, -11.5, info, color='#88aacc', fontsize=8.5,
            ha='center', va='bottom', family='monospace',
            bbox=dict(facecolor='#050a14', alpha=0.7, pad=5,
                      edgecolor='#1a3060', linewidth=0.5))

    #  Labels
    ax.text(r_ph + 0.2, 0.15, 'photon sphere',
            color='orange', fontsize=8, alpha=0.6)
    ax.text(r_isco * 0.68, r_isco * 0.68 * 0.35 + 0.3, 'ISCO',
            color='red', fontsize=8, alpha=0.6)

    ax.set_title(f'Kerr Black Hole (spin a = {spin:.2f})',
                 color='white', fontsize=12, pad=10)
    plt.tight_layout()
    plt.show()


interact(draw_black_hole_snapshot)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.style.use('dark_background')

# Simulation parameters
M    = 1.5
spin = 0.8    # high spin — strong jets
r_s  = schwarzschild_radius(M)
r_isco = isco_radius(M, spin)
r_ph = photon_sphere_radius(M)
N    = 130    # disk particles

np.random.seed(99)
radii  = np.random.uniform(r_isco * 1.05, r_isco * 5.5, N)
angles = np.random.uniform(0, 2*np.pi, N)
history_x = [[] for _ in range(N)]
history_y = [[] for _ in range(N)]
TRAIL = 20

# Background stars (will be lensed)
# 200 random "stars" — dots in the background.
# they are to the black hole. Closer = more deflection.
N_STARS = 220
star_true_x = np.random.uniform(-15, 15, N_STARS)  # true positions
star_true_y = np.random.uniform(-15, 15, N_STARS)
star_bright  = np.random.uniform(0.2, 0.9, N_STARS)
star_size    = np.random.uniform(0.5, 2.5, N_STARS)

def lens_position(tx, ty, M, r_s):
    """
    Gravitational lensing: deflect apparent star position.

    The true deflection angle from GR is:
        alpha = 4GM / (c² * b)
    where b = impact parameter (closest approach distance).

    For our 2D simulation we approximate this by pushing
    the apparent position AWAY from the black hole center,
    proportional to 1/distance². This creates the same
    visual effect as the real Einstein ring distortion.
    """
    dist = np.sqrt(tx**2 + ty**2)
    if dist < r_s * 1.2:
        return None, None   # behind event horizon — invisible

    # Deflection strength drops off as 1/dist²
    # The 4*r_s factor comes from the GR deflection formula
    deflect = 4 * r_s / (dist + 0.001)
    deflect = np.clip(deflect, 0, 0.85)   # cap to avoid infinite deflection

    # Push apparent position outward (away from black hole)
    # This mimics how lensing smears light around a massive object
    ax_pos = tx + tx * deflect / dist * deflect
    ay_pos = ty + ty * deflect / dist * deflect
    return ax_pos, ay_pos

# ── Jet particles ──────────────────────────────────────────
# Jets shoot out along the rotation axis (vertical in our view).
# Each jet particle has:
#   - position (x, y)
#   - velocity (vx, vy) — mostly vertical, slight lateral spread
#   - age — how long it has existed
N_JET = 90
jet_x   = np.zeros(N_JET)
jet_y   = np.zeros(N_JET)
jet_vx  = np.random.uniform(-0.08, 0.08, N_JET)  # slight spread
jet_vy  = np.random.uniform(0.3, 0.9, N_JET)     # upward speed
jet_age = np.random.randint(0, 60, N_JET)         # stagger start times
jet_side = np.where(np.arange(N_JET) < N_JET//2, 1, -1)  # up or down

JET_MAXAGE = 70   # frames before particle respawns

# ── Figure setup ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 10))
ax.set_facecolor('#000008')
ax.set_xlim(-15, 15)
ax.set_ylim(-18, 18)
ax.set_aspect('equal')
ax.axis('off')
fig.tight_layout()

# Pre-create plot objects (faster than recreating each frame)
# Stars
star_plot = ax.scatter([], [], c=[], s=[], alpha=0.7, zorder=1)

# Disk particle trails and dots
trail_lines = [ax.plot([], [], '-', lw=1.1, alpha=0)[0] for _ in range(N)]
disk_dots   = [ax.plot([], [], 'o', ms=2.5, alpha=0)[0] for _ in range(N)]

# Jet particles
jet_plot = ax.scatter([], [], c=[], s=[], alpha=0.8, zorder=6)

# Static black hole elements (drawn once)
for i in range(8, 0, -1):
    ax.add_patch(plt.Circle((0,0), r_ph*(1+i*0.22),
                             color='#ff5500', alpha=0.013*i, zorder=3))
ax.add_patch(plt.Circle((0,0), r_ph, fill=False,
                         edgecolor='#ff8800', lw=1.2, alpha=0.4, zorder=4))
ax.add_patch(plt.Circle((0,0), r_s, color='black', zorder=12))

# HUD text
hud = ax.text(0, -17, '', color='#6688aa', fontsize=8,
              ha='center', va='bottom', family='monospace',
              bbox=dict(facecolor='#020810', alpha=0.8, pad=4,
                        edgecolor='#0a2040', linewidth=0.5))

frame_num = [0]

def update(frame):
    frame_num[0] += 1
    t = frame_num[0]

    # 1. UPDATE DISK PARTICLES
    for i in range(N):
        omega = keplerian_omega(M, radii[i])
        angles[i] += omega * 0.22

        # Spiral inward (accretion)
        radii[i] -= 0.0015 * (r_isco / max(radii[i], r_s))

        # Respawn if swallowed
        if radii[i] < r_s:
            radii[i] = np.random.uniform(r_isco * 2, r_isco * 5.5)
            angles[i] = np.random.uniform(0, 2*np.pi)
            history_x[i].clear()
            history_y[i].clear()
            continue

        x = radii[i] * np.cos(angles[i])
        y = radii[i] * np.sin(angles[i]) * 0.35

        history_x[i].append(x)
        history_y[i].append(y)
        if len(history_x[i]) > TRAIL:
            history_x[i].pop(0)
            history_y[i].pop(0)

        temp  = disk_temperature(radii[i], M)
        color = plt.cm.afmhot(temp)

        trail_lines[i].set_data(history_x[i], history_y[i])
        trail_lines[i].set_color(color)
        trail_lines[i].set_alpha(0.3)

        disk_dots[i].set_data([x], [y])
        disk_dots[i].set_color(color)
        disk_dots[i].set_alpha(0.9)

    # 2. UPDATE JET PARTICLES
    # Jets are stronger with higher spin — we multiply by spin²
    # The Blandford-Znajek power scales as a² (spin squared)
    jet_power = spin ** 2

    for i in range(N_JET):
        jet_age[i] += 1

        if jet_age[i] > JET_MAXAGE:
            # Respawn near the black hole center
            jet_x[i] = np.random.uniform(-r_s*0.3, r_s*0.3)
            jet_y[i] = 0.0
            jet_vx[i] = np.random.uniform(-0.06, 0.06)
            # Jet speed proportional to spin (Kerr black hole)
            jet_vy[i] = np.random.uniform(0.25, 0.85) * jet_power * 1.8
            jet_age[i] = 0

        # Move jet particle
        jet_x[i] += jet_vx[i] * jet_power
        jet_y[i] += jet_vy[i] * jet_side[i] * jet_power

    # Color jets: blue-white (relativistic plasma)
    # Age 0 = bright white-blue, older = dimmer blue-purple
    jet_ages_norm = np.clip(jet_age / JET_MAXAGE, 0, 1)
    jet_colors = plt.cm.cool(1 - jet_ages_norm)
    jet_sizes  = (1 - jet_ages_norm) * 18 + 2

    jet_plot.set_offsets(np.column_stack([jet_x, jet_y]))
    jet_plot.set_color(jet_colors)
    jet_plot.set_sizes(jet_sizes)
    jet_plot.set_zorder(6)

    #3. UPDATE LENSED STARS
    app_x, app_y, s_colors, s_sizes = [], [], [], []
    for j in range(N_STARS):
        lx, ly = lens_position(star_true_x[j], star_true_y[j], M, r_s)
        if lx is None:
            continue   # star hidden behind black hole

        # Stars near the lensing zone get brighter (Einstein ring effect)
        dist_from_center = np.sqrt(star_true_x[j]**2 + star_true_y[j]**2)
        lens_boost = 1 + 3 * r_ph / max(dist_from_center, r_ph)

        app_x.append(lx)
        app_y.append(ly)
        s_colors.append([1.0, 1.0, 0.95, star_bright[j] * min(lens_boost, 1.8)])
        s_sizes.append(star_size[j] * lens_boost * 0.9)

    if app_x:
        star_plot.set_offsets(np.column_stack([app_x, app_y]))
        star_plot.set_color(s_colors)
        star_plot.set_sizes(s_sizes)

    #4. UPDATE HUD
    td = gravitational_time_dilation(r_isco, M)
    v  = orbital_speed(M, r_isco)
    hud.set_text(
        f"M={M:.1f}  spin a={spin:.2f}  r_s={r_s:.2f}  "
        f"r_isco={r_isco:.2f}\n"
        f"v_isco={v:.3f}c   time dilation={td:.3f}×   "
        f"jet power={jet_power:.2f}"
    )

    return trail_lines + disk_dots + [jet_plot, star_plot, hud]

anim = FuncAnimation(fig, update, frames=500,
                     interval=28, blit=True)
HTML(anim.to_jshtml())